[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/next-gen-aiam/blob/main/notebooks/05_dl_portfolio_construction_exploration.ipynb)

# Notebook 05 — Deep Learning Portfolio Construction (Direct-Weight Walk-Forward)

*Brandt-Santa-Clara-Valkanov (2009) parametric portfolio policies on a 2-asset SPY+IEF
universe with 252-day raw return + realized-power lag features and monthly walk-forward refit.*

## Abstract

This notebook applies the Brandt, Santa-Clara & Valkanov (2009) parametric portfolio policy framework to a 2-asset universe (SPY + IEF) using the BSV (2009) direct-weight framework: 252-lag raw return history (and optionally raw realized-power history) as features, monthly walk-forward refit with 24-month rolling training windows, and three portfolio-level losses (Sharpe, CRRA-γ5, CRRA+shrinkage).  The central question: does adding intraday-derived realized power to the 252-lag history improve out-of-sample Sharpe over daily returns alone?

We test 3 architectures × 2 feature variants × 3 losses = **18 configs × 10 seeds × 22 refits = 3,960 policy fits** (Colab Blackwell, ~3-4 hours).  A prior simpler version of this notebook used engineered features and a single fit; its null result (best DL Sharpe 1.164 vs benchmark 1.167) motivates testing two alternative hypotheses simultaneously here: H1 = richer raw-history features, H2 = walk-forward methodology adaptation.

## Chapter Context — End-to-End DL as a Portfolio Optimizer

This notebook is the **third leg** of the cross-paradigm DL investigation and is best read alongside Notebook 04 (predict-then-optimize DL) and the master comparison table in Notebook 00.

**What this tests.** Notebook 04 runs a *predict-then-optimize* pipeline (DL → μ̂ → MSR). That design amplifies estimation error at the optimizer step — a well-known pathology. This notebook tests the complementary paradigm: *direct-weight* (end-to-end) optimization following Brandt, Santa-Clara & Valkanov (BSV 2009), where the network outputs portfolio weights directly and is trained on a portfolio-level loss (Sharpe or CRRA utility). By bypassing the optimizer, DL gets its fairest possible test.

**The finding.** The result is null. Best DL Sharpe **~1.240** vs Risk Parity **1.247** (gap −0.008). This corroborates Notebook 04's conclusion and completes the cross-paradigm picture.

**Three-leg narrative — complexity does not pay:**

| Notebook | Paradigm | Result |
|---|---|---|
| 03 | ML ensemble (Lasso/RF/XGB → MSR) | **Wins** — Sharpe 2.579, beats all classical baselines |
| 04 | DL predict-then-optimize (DL → μ̂ → MSR) | **Misses** — best 2.386, gap −0.193 vs ML bar |
| 05 | DL end-to-end (direct-weight, BSV 2009) | **Ties** trivial Risk Parity benchmark (−0.008) |

**Honest caveat on scope.** A 2-asset universe (SPY + IEF) is the *least* favorable setting for DL: minimal cross-sectional signal, highly liquid mean-reverting assets. The 29-asset cross-asset extension sketched in §5.4 is the stronger, as-yet-untested case. The null here is informative but not the last word.

## §T1 Theoretical Framework — BSV (2009)

### Direct Portfolio Policy Parameterization

Brandt, Santa-Clara & Valkanov (2009) propose mapping asset characteristics directly to portfolio weights without an intermediate forecasting step:

$$w_{i,t} = f_\theta(x_{i,t})$$

where $x_{i,t}$ is the vector of observable characteristics for asset $i$ at time $t$, and $\theta$ are the policy parameters. The portfolio objective is maximized end-to-end:

$$\max_\theta \frac{1}{T}\sum_{t=1}^{T} U(R_{p,t+1}), \qquad R_{p,t+1} = \sum_{i=1}^{N} w_{i,t}(\theta)\, r_{i,t+1}$$

This directly avoids the noise amplification of mean-variance optimization on small-sample returns.

**Reference:** Brandt, M. W., Santa-Clara, P., & Valkanov, R. (2009). Parametric portfolio policies: Exploiting characteristics in the cross-section of equity returns. *Review of Financial Studies*, 22(9), 3411–3447.

## §T2 Loss Functions

We compare three portfolio-level training objectives:

### Sharpe Loss

$$\mathcal{L}_{\text{Sharpe}}(\theta) = -\frac{\hat{\mu}(R_p)}{\hat{\sigma}(R_p)}$$

### CRRA Utility Loss (γ = 5)

$$\mathcal{L}_{\text{CRRA}}(\theta) = -\frac{1}{T}\sum_{t=1}^{T} \frac{(1 + R_{p,t})^{1-\gamma}}{1-\gamma}$$

γ=5 corresponds to moderately high risk aversion (a standard moderately-high risk-aversion default).

### CRRA + Shrinkage to Risk Parity

$$w_{i,t} = \sigma(f_\theta(x_{i,t})) \cdot w_i^{\text{RP}}$$

where $\sigma(\cdot)$ is the sigmoid function and $w^{\text{RP}}$ is the 21-day vol-weighted risk-parity benchmark weight.

**Reference:** Brandt, M. W. (1999). Estimating portfolio and consumption choice: A conditional Euler equations approach. *Journal of Finance*, 54(5), 1609–1645.

## §T3 Feature Design — 252-Lag Raw History

### Return lags

Each observation $(t, \text{asset})$ carries 252 lagged daily log-returns: $[r_{t}, r_{t-1}, \ldots, r_{t-251}]$.  The entire history is the feature vector — no hand-crafted momentum or volatility statistics.

### Realized power (optional second channel)

$$\text{RP}_t = \sum_k r_{t,k}^2$$

where $r_{t,k}$ are intraday 5-minute log returns within trading day $t$ (09:30–16:00 ET, ~78 bars).  In the `daily+RP` variant the 252 RP lags are appended after the 252 return lags, producing a 504-column flat feature vector.  For sequence models (LSTM, Transformer) this is reshaped to $(252, 2)$ — timestep × channel — so each timestep carries a (return, RP) pair.

### Architecture-specific reshaping

- **MLP**: flat $(\text{batch}, 252)$ or $(\text{batch}, 504)$
- **LSTM / Transformer**: reshaped to $(\text{batch}, 252, n_{\text{ch}})$ where $n_{\text{ch}} \in \{1, 2\}$

## §T4 What This Exploration Adds

This is a **full replication** of the BSV (2009) direct-weight framework on our SPY+IEF universe, distinct from an earlier engineered-feature single-fit version whose null result (best DL Sharpe 1.164 vs benchmark 1.167) motivated the methodology upgrade.

Key differences from that earlier version:
- **252-day raw return + RP lag history** instead of 6 engineered features
- **24-month rolling training window, monthly refit** (22 refit events: 2024-07 → 2026-04) instead of single fit
- **3 architectures × 2 feature variants × 3 losses = 18 configs × 10 seeds × 22 refits = 3,960 fits**
- **Compute**: Colab Blackwell, ~3-4 hours; built on infrastructure from commit `7386d17`
- **Memory-managed**: one config at a time; progress saved to `results/notebook_05/config_progress.csv` after each config


---
## §0 Setup

In [ ]:
import os, sys, time, warnings
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print('Detected Colab — setting up...')

    REPO_DIR = Path('/content/next-gen-aiam')

    # Reset to /content (handles reused runtime where CWD is already inside repo)
    os.chdir('/content')

    if not REPO_DIR.exists():
        print('Cloning repo...')
        os.system('git clone https://github.com/FranQuant/next-gen-aiam.git 2>&1 | tail -3')
    else:
        print(f'Repo already cloned at {REPO_DIR}; pulling latest...')
        os.system(f'cd {REPO_DIR} && git pull --rebase 2>&1 | tail -3')

    # chdir into repo using absolute path (relative chdir causes nesting on rerun)
    os.chdir(str(REPO_DIR))
    print(f'Working dir: {os.getcwd()}')

    print('Installing dependencies (1-2 min)...')
    os.system('pip install -q -e . 2>&1 | tail -3')
    os.system('pip install -q xgboost pyarrow scipy scikit-learn cvxpy pandas-market-calendars 2>&1 | tail -3')
else:
    print(f'Detected local environment: {os.getcwd()}')

import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    GPU_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    GPU_NAME = 'Apple Silicon MPS'
else:
    DEVICE = 'cpu'
    GPU_NAME = 'CPU'

RUN_MODE = 'full' if (IS_COLAB and DEVICE == 'cuda') else 'smoke'

ASSETS = ['SPY.US', 'IEF.US']
LOOKBACK = 252
TEST_START = '2024-07-01'
TEST_END   = '2026-04-30'
TRADING_DAYS = 252
SMOKE_N_REFITS = 3

if RUN_MODE == 'full':
    SEEDS = tuple(range(10))
    MAX_EPOCHS = 80
    PATIENCE = 12
else:
    SEEDS = (0,)
    MAX_EPOCHS = 10
    PATIENCE = 4

print(f'Device: {DEVICE} ({GPU_NAME})')
print(f'torch: {torch.__version__}')
print(f'Run mode: {RUN_MODE}')
print(f'Seeds: {len(SEEDS)}  |  Max epochs: {MAX_EPOCHS}  |  Patience: {PATIENCE}')

GPU check — confirms the CUDA device on Colab; harmless no-op locally.

In [ ]:
!nvidia-smi || echo "No NVIDIA GPU — local CPU/MPS run"

## §1 Imports

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "figure.dpi": 120,
})
warnings.filterwarnings('ignore')

ROOT = Path(os.getcwd())
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

for _p in [str(ROOT / 'src'), str(ROOT / 'notebooks')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from _shared import ann_sharpe, ann_return, ann_vol, max_drawdown, TRADING_DAYS
from aiam.data.intraday import load_cached_intraday
from aiam.features.realized_power import compute_realized_power_5min
from aiam.features.raw_history import extract_raw_history, get_raw_feature_cols
from aiam.dl.losses import sharpe_loss, crra_loss, crra_shrinkage_loss
from aiam.dl.policies import MLPPolicy, LSTMPolicy, TransformerPolicy
from aiam.dl.policy_workflow import DirectWeightSeedEnsemble
from aiam.dl.walkforward import WalkForwardEnsemble, generate_refit_dates, fit_walkforward_direct_weight
import torch.nn as nn

OUT_DIR = ROOT / 'results' / 'notebook_05'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

## §2 Data Loading

In [ ]:
prices_all = pd.read_parquet(ROOT / 'data/cache/prices_29.parquet')
daily_prices = prices_all[ASSETS].copy()
daily_prices.index = pd.to_datetime(daily_prices.index)
daily_prices = daily_prices.loc['2021-01-04':'2026-04-30']
print(f'Daily prices: {daily_prices.shape}  |  {daily_prices.index.min().date()} -> {daily_prices.index.max().date()}')

daily_returns = daily_prices.pct_change().dropna()
print(f'Daily returns: {daily_returns.shape}')

# Intraday: cache parquet is gitignored; build from published CSV on Colab
intraday_cache = ROOT / 'data/cache/intraday_5min_SPY_IEF_2021_2026.parquet'
intraday_published_csv = ROOT / 'data/published/intraday_5min_SPY_IEF_2021_2026.csv'

if not intraday_cache.exists():
    print('Parquet cache not found; building from published CSV...')
    intraday_cache.parent.mkdir(parents=True, exist_ok=True)
    intraday_df = pd.read_csv(intraday_published_csv)
    intraday_df['timestamp'] = pd.to_datetime(intraday_df['timestamp'], utc=True)
    intraday_df = intraday_df.set_index(['timestamp', 'asset']).sort_index()
    intraday_df.to_parquet(intraday_cache)
    print(f'Built parquet at {intraday_cache} ({intraday_cache.stat().st_size / 1e6:.1f} MB)')

intraday = load_cached_intraday(intraday_cache, tickers=ASSETS)
print(f'Intraday bars: {intraday.shape}')

## §3 Realized Power (Daily)

In [ ]:
rp_wide = compute_realized_power_5min(intraday, tickers=ASSETS)
rp_wide.index = pd.to_datetime(rp_wide.index)
print(f'Realized power wide: {rp_wide.shape}')
print(f'Date range: {rp_wide.index.min().date()} -> {rp_wide.index.max().date()}')
print(rp_wide.head(3))

## §4 Build Raw History Feature Panels

In [ ]:
# Align RP to daily_returns index before passing to extract_raw_history
rp_aligned = rp_wide.reindex(daily_returns.index)

panel_daily_only = extract_raw_history(
    returns_wide=daily_returns,
    realized_power_wide=None,
    lookback=LOOKBACK,
)
panel_daily_rp = extract_raw_history(
    returns_wide=daily_returns,
    realized_power_wide=rp_aligned,
    lookback=LOOKBACK,
)

cols_daily_only = get_raw_feature_cols(lookback=LOOKBACK, include_rp=False)
cols_daily_rp   = get_raw_feature_cols(lookback=LOOKBACK, include_rp=True)

# Internal keys: 'daily' and 'dailyrp' (no '+' for filename safety)
FEATURE_PANELS = {'daily': panel_daily_only, 'dailyrp': panel_daily_rp}
FEATURE_COLS   = {'daily': cols_daily_only,  'dailyrp': cols_daily_rp}
N_CHANNELS     = {'daily': 1,                'dailyrp': 2}
FEAT_DISPLAY   = {'daily': 'daily-only',     'dailyrp': 'daily+RP'}

d_min = panel_daily_only.index.get_level_values(0).min()
d_max = panel_daily_only.index.get_level_values(0).max()
print(f'panel_daily_only: {panel_daily_only.shape}  ({d_min.date()} -> {d_max.date()})')
print(f'panel_daily_rp:   {panel_daily_rp.shape}')
print(f'Daily-only: {len(cols_daily_only)} cols  |  Daily+RP: {len(cols_daily_rp)} cols')

## §5 Target Construction and Walk-Forward Refit Dates

In [ ]:
# 1-day forward returns (daily rebalance per BSV convention)
fwd_ret_wide = daily_returns.shift(-1).dropna()
target_series = fwd_ret_wide.stack()
target_series.index.names = ['date', 'asset']
target_series.name = 'fwd_ret_1d'

# Test period slices
test_dates = daily_returns.loc[TEST_START:TEST_END].index
test_daily_ret = daily_returns.loc[test_dates, ASSETS]
print(f'Test period: {test_dates.min().date()} -> {test_dates.max().date()}  ({len(test_dates)} days)')

# Walk-forward refit schedule
all_refit_dates = generate_refit_dates(
    test_start=pd.Timestamp(TEST_START),
    test_end=pd.Timestamp(TEST_END),
    cadence='monthly',
)
print(f'Full refit schedule: {len(all_refit_dates)} dates  ({all_refit_dates[0].date()} -> {all_refit_dates[-1].date()})')

# Active refits (smoke = first 3 only)
active_refit_dates = all_refit_dates[:SMOKE_N_REFITS] if RUN_MODE == 'smoke' else all_refit_dates
print(f'Active refits ({RUN_MODE}): {len(active_refit_dates)}')
print(f'Training window: 24 months rolling, 8-day quarantine, 15% val split')

## §6 Benchmark Strategies

In [ ]:
ret_ew = test_daily_ret.mean(axis=1)
ret_ew.name = 'EW-50/50'

ret_6040 = test_daily_ret['SPY.US'] * 0.60 + test_daily_ret['IEF.US'] * 0.40
ret_6040.name = '60/40'

roll_vol = daily_returns.rolling(21).std().shift(1)
inv_vol = 1.0 / roll_vol.clip(lower=1e-8)
rp_weights_wide = inv_vol.div(inv_vol.sum(axis=1), axis=0)
ret_rp = (test_daily_ret * rp_weights_wide.reindex(test_dates)).sum(axis=1)
ret_rp.name = 'RiskParity-21d'

rp_benchmark_w = rp_weights_wide.reindex(test_dates)[ASSETS].mean().values.astype('float32')

BENCHMARKS = {'RiskParity-21d': ret_rp, '60/40': ret_6040, 'EW-50/50': ret_ew}

print('Benchmark stats (test period):')
for bname, bret in BENCHMARKS.items():
    print(f'  {bname:20s}  Sharpe={ann_sharpe(bret):.3f}  AnnRet={ann_return(bret):.1%}  MaxDD={max_drawdown(bret):.1%}')

## §7 Sequence-Model Wrappers + Walk-Forward Training

The `fit_walkforward_direct_weight` infrastructure expects flat `(batch, n_features)` arrays.
LSTM and Transformer policies need `(batch, seq, channels)` input.  We define notebook-level
wrapper classes that reshape inside `forward()` — no changes to `src/aiam/`.

For `daily+RP`, flat layout is `[r_lag_0...r_lag_251, rp_lag_0...rp_lag_251]`.
Reshape to `(batch, n_ch=2, 252)` then permute to `(batch, 252, 2)` — correct channel ordering.

In [ ]:
def make_seq_policy(inner_cls, lookback, n_channels, extra_init_kwargs=None):
    """Return a flat-input wrapper around inner_cls for sequence models.

    The wrapper reshapes (batch, lookback*n_channels) -> (batch, lookback, n_channels)
    inside forward(). n_features kwarg from fit_walkforward is absorbed and ignored;
    the inner model receives n_features=n_channels (the per-timestep channel count).
    """
    _extra = extra_init_kwargs or {}

    class _Policy(nn.Module):
        def __init__(self, n_features, n_assets, **kwargs):  # n_features absorbed here
            super().__init__()
            merged = {**_extra, **kwargs}
            self._inner = inner_cls(n_features=n_channels, n_assets=n_assets, **merged)
            self._lk = lookback
            self._nc = n_channels

        def forward(self, x):
            b = x.shape[0]
            return self._inner(x.reshape(b, self._nc, self._lk).permute(0, 2, 1))

    _Policy.__name__ = f'Reshaped{inner_cls.__name__}'
    return _Policy


STRATEGY_RETURNS   = {}   # config_key -> pd.Series
STRATEGY_WEIGHTS   = {}   # config_key -> pd.DataFrame(date x asset)
STABILITY_PER_SEED = {}   # config_key -> list[float] (per-seed OOS Sharpes)


def _infer_ensemble(wf_ens, feature_panel, feature_cols, test_X):
    """Run ensemble inference for all test dates. Returns (rets_dict, w_rows)."""
    rets_dict, w_rows = {}, []
    for d in test_dates:
        X_d = test_X.get(d)
        if X_d is None:
            w = np.array([0.5, 0.5])
        else:
            try:
                raw = wf_ens.predict_weights_for_date(X_d, d)
                w = np.clip(raw.mean(axis=0), 0, None)
                w = w / max(w.sum(), 1e-12)
            except (ValueError, RuntimeError):
                w = np.array([0.5, 0.5])
        r = test_daily_ret.loc[d].reindex(ASSETS).to_numpy()
        rets_dict[d] = float(np.dot(w, r))
        w_rows.append({ASSETS[0]: float(w[0]), ASSETS[1]: float(w[1])})
    return rets_dict, w_rows


def _per_seed_stability(wf_ens, test_X):
    """Compute per-seed OOS Sharpes across walk-forward refits."""
    per_seed_sh = []
    for si in range(len(SEEDS)):
        seed_rets = []
        for d in test_dates:
            X_d = test_X.get(d)
            if X_d is None:
                seed_rets.append(np.nan)
                continue
            try:
                ref_ens = wf_ens.ensemble_for_date(d)
                single = DirectWeightSeedEnsemble(fits=[ref_ens.fits[si]], seeds=(SEEDS[si],))
                raw = single.predict_weights(X_d)
                w = np.clip(raw.mean(axis=0), 0, None)
                w = w / max(w.sum(), 1e-12)
                r = test_daily_ret.loc[d].reindex(ASSETS).to_numpy()
                seed_rets.append(float(np.dot(w, r)))
            except Exception:
                seed_rets.append(np.nan)
        per_seed_sh.append(ann_sharpe(pd.Series(seed_rets)))
    return per_seed_sh


def _save_progress(config_key, arch, fv, lv, per_seed_sh, ret_s):
    prog_file = OUT_DIR / 'config_progress.csv'
    row = {
        'config': config_key, 'arch': arch, 'feat_variant': FEAT_DISPLAY[fv], 'loss': lv,
        'sharpe_ens': ann_sharpe(ret_s),
        'sharpe_mean': np.nanmean(per_seed_sh),
        'sharpe_std': np.nanstd(per_seed_sh),
        'n_seeds': len(SEEDS),
        'n_refits': len(active_refit_dates),
    }
    existing = pd.read_csv(prog_file) if prog_file.exists() else pd.DataFrame()
    pd.concat([existing, pd.DataFrame([row])], ignore_index=True).to_csv(prog_file, index=False)


def _run_config(arch_name, fv, lv, t0_global):
    config_key = f'{arch_name}_{fv}_{lv}'
    feature_panel = FEATURE_PANELS[fv]
    feature_cols  = FEATURE_COLS[fv]
    n_ch = N_CHANNELS[fv]
    n_flat = len(feature_cols)
    activation = 'sigmoid' if lv == 'crra_shrinkage' else 'relu'

    if arch_name == 'MLP':
        policy_cls   = MLPPolicy
        policy_extra = dict(hidden_dims=(32, 16), dropout=0.10)
    elif arch_name == 'LSTM':
        policy_cls   = make_seq_policy(LSTMPolicy, LOOKBACK, n_ch)
        policy_extra = dict(hidden_dim=24, dropout=0.10)
    else:
        policy_cls   = make_seq_policy(
            TransformerPolicy, LOOKBACK, n_ch,
            extra_init_kwargs={'max_lookback': LOOKBACK},
        )
        policy_extra = dict(d_model=32, nhead=4, num_layers=2, dropout=0.10)

    t1 = time.time()
    wf_ens = fit_walkforward_direct_weight(
        feature_panel=feature_panel,
        target_panel=target_series,
        feature_cols=feature_cols,
        assets=ASSETS,
        refit_dates=active_refit_dates,
        policy_class=policy_cls,
        loss_kind=lv,
        seeds=SEEDS,
        training_window_months=24,
        validation_share=0.15,
        quarantine_days=8,
        benchmark_w=rp_benchmark_w if lv == 'crra_shrinkage' else None,
        device=DEVICE,
        verbose=False,
        n_features=n_flat, n_assets=len(ASSETS), activation=activation,
        lr=1e-3, weight_decay=1e-4, batch_size=256,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        **policy_extra,
    )
    fit_secs = time.time() - t1

    # Precompute test-period feature rows once (avoids repeated .xs() calls)
    test_X = {}
    for d in test_dates:
        try:
            X_d = feature_panel.xs(d, level='date')[feature_cols].reindex(ASSETS)
            if not X_d.isna().any().any():
                test_X[d] = X_d.values.astype('float32')
        except KeyError:
            pass

    # Ensemble inference
    rets_dict, w_rows = _infer_ensemble(wf_ens, feature_panel, feature_cols, test_X)
    STRATEGY_RETURNS[config_key]  = pd.Series(rets_dict, name=config_key)
    STRATEGY_WEIGHTS[config_key]  = pd.DataFrame(w_rows, index=list(rets_dict.keys()))

    # Per-seed stability
    per_seed_sh = _per_seed_stability(wf_ens, test_X)
    STABILITY_PER_SEED[config_key] = per_seed_sh

    del wf_ens  # free memory before next config

    sh_ens = ann_sharpe(STRATEGY_RETURNS[config_key])
    elapsed = time.time() - t0_global
    print(
        f'  [{elapsed:5.0f}s] {config_key:42s}  '
        f'Sharpe={sh_ens:.3f}  '
        f'mean+/-std={np.nanmean(per_seed_sh):.3f}+/-{np.nanstd(per_seed_sh):.3f}  '
        f'fit={fit_secs:.0f}s'
    )
    _save_progress(config_key, arch_name, fv, lv, per_seed_sh, STRATEGY_RETURNS[config_key])


# ── Main loop ────────────────────────────────────────────────────────────────
t0_global = time.time()
configs = list(itertools.product(
    ['MLP', 'LSTM', 'Transformer'],
    list(FEATURE_PANELS.keys()),
    ['sharpe', 'crra', 'crra_shrinkage'],
))
n_fits = len(configs) * len(SEEDS) * len(active_refit_dates)
print(f'Running {len(configs)} configs x {len(SEEDS)} seeds x {len(active_refit_dates)} refits = {n_fits} policy fits')
print(f'Mode: {RUN_MODE}  |  Device: {DEVICE}')
print()

for arch, fv, lv in configs:
    _run_config(arch, fv, lv, t0_global)

print(f'\nAll {len(configs)} configs done in {(time.time()-t0_global)/60:.1f} min')

## §8 Per-Seed Stability Table

In [ ]:
stability_rows = []
for config_key, per_seed_sh in STABILITY_PER_SEED.items():
    parts = config_key.split('_', 2)
    arch, fv, lv = parts[0], parts[1], parts[2]
    stability_rows.append({
        'config': config_key,
        'arch': arch,
        'feat_variant': FEAT_DISPLAY[fv],
        'loss': lv,
        'sharpe_mean': np.nanmean(per_seed_sh),
        'sharpe_std':  np.nanstd(per_seed_sh),
        'sharpe_min':  np.nanmin(per_seed_sh) if per_seed_sh else np.nan,
        'sharpe_max':  np.nanmax(per_seed_sh) if per_seed_sh else np.nan,
        'per_seed_sharpes': per_seed_sh,
    })

stability_df = pd.DataFrame(stability_rows)
print(stability_df[['config', 'sharpe_mean', 'sharpe_std', 'sharpe_min', 'sharpe_max']].to_string(index=False))

## §9 Comparison Table — 21 Strategies (18 DL + 3 Benchmarks)

In [ ]:
comp_rows = []
for config_key, ret_s in STRATEGY_RETURNS.items():
    parts = config_key.split('_', 2)
    arch, fv, lv = parts[0], parts[1], parts[2]
    comp_rows.append({
        'strategy': config_key,
        'arch': arch, 'feat_variant': FEAT_DISPLAY[fv], 'loss': lv,
        'family': f'DL-{FEAT_DISPLAY[fv]}',
        'sharpe':   ann_sharpe(ret_s),
        'ann_ret':  ann_return(ret_s),
        'ann_vol':  ann_vol(ret_s),
        'max_dd':   max_drawdown(ret_s),
    })

for bname, bret in BENCHMARKS.items():
    comp_rows.append({
        'strategy': bname, 'arch': 'Benchmark', 'feat_variant': '-', 'loss': '-',
        'family': 'Benchmark',
        'sharpe':  ann_sharpe(bret), 'ann_ret': ann_return(bret),
        'ann_vol': ann_vol(bret),    'max_dd':  max_drawdown(bret),
    })

comp_df = pd.DataFrame(comp_rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
comp_df['rank'] = comp_df.index + 1

print(f'Total strategies: {len(comp_df)}')
print(comp_df[['rank', 'strategy', 'sharpe', 'ann_ret', 'ann_vol', 'max_dd']].to_string(index=False))
comp_df.to_csv(OUT_DIR / 'comparison.csv', index=False)
print(f'\nSaved -> {OUT_DIR / "comparison.csv"}')

## §10 Plots

In [ ]:
# ── Figure 1: Equity Curves — Top-5 DL + 3 Benchmarks ──────────────────────
top5_names  = comp_df[comp_df['arch'] != 'Benchmark']['strategy'].head(5).tolist()
bench_names = list(BENCHMARKS.keys())
colors_dl = plt.cm.tab10(np.linspace(0, 0.5, 5))
colors_bm = ['#333333', '#888888', '#bbbbbb']

fig, ax = plt.subplots(figsize=(12, 5))
for i, name in enumerate(top5_names):
    cum = (1 + STRATEGY_RETURNS[name].dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=colors_dl[i], lw=1.5, label=name)
for j, bname in enumerate(bench_names):
    cum = (1 + BENCHMARKS[bname].dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=colors_bm[j], lw=1.5, ls='--', label=bname)
ax.set_yscale('log')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Wealth (log scale)')
ax.set_title('Equity Curves — Top-5 DL Walk-Forward Strategies vs Benchmarks (Test Period)')
ax.legend(fontsize=7, ncol=2, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'equity_curves_05.png', dpi=150)
plt.show()
print('Saved equity_curves_05.png')

In [ ]:
# ── Figure 2: Drawdown ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
for i, name in enumerate(top5_names):
    r = STRATEGY_RETURNS[name].dropna()
    cum = (1 + r).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=colors_dl[i])
    ax.plot(dd.index, dd.values, color=colors_dl[i], lw=1, label=name)
for j, bname in enumerate(bench_names):
    r = BENCHMARKS[bname].dropna()
    cum = (1 + r).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.12, color=colors_bm[j])
    ax.plot(dd.index, dd.values, color=colors_bm[j], lw=1, ls='--', label=bname)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.set_title('Underwater Drawdown — Top-5 DL Walk-Forward Strategies vs Benchmarks')
ax.legend(fontsize=7, ncol=2, loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'drawdown_05.png', dpi=150)
plt.show()
print('Saved drawdown_05.png')

In [ ]:
# ── Figure 3: Walk-Forward Weight Time Series — Transformer (6-panel) ────────
FEAT_KEYS = list(FEATURE_PANELS.keys())  # ['daily', 'dailyrp']
LOSS_KEYS = ['sharpe', 'crra', 'crra_shrinkage']

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
for ri, fv in enumerate(FEAT_KEYS):
    for ci, lv in enumerate(LOSS_KEYS):
        ax = axes[ri][ci]
        name = f'Transformer_{fv}_{lv}'
        if name not in STRATEGY_WEIGHTS:
            ax.set_title(f'{FEAT_DISPLAY[fv]} + {lv}\n(missing)', fontsize=8)
            continue
        wdf = STRATEGY_WEIGHTS[name]
        ax.stackplot(wdf.index, wdf[ASSETS[0]].values, wdf[ASSETS[1]].values,
                     labels=['SPY', 'IEF'], colors=['#1f77b4', '#ff7f0e'], alpha=0.8)
        lv_label = {'sharpe': 'Sharpe', 'crra': 'CRRA', 'crra_shrinkage': 'CRRA+Shrink'}[lv]
        ax.set_title(f'{FEAT_DISPLAY[fv]} | {lv_label}', fontsize=9)
        ax.set_ylim(0, 1)
        ax.set_ylabel('Weight', fontsize=8)
        ax.tick_params(axis='x', labelsize=7, rotation=30)

spy_patch = mpatches.Patch(color='#1f77b4', alpha=0.8, label='SPY')
ief_patch = mpatches.Patch(color='#ff7f0e', alpha=0.8, label='IEF')
fig.legend(handles=[spy_patch, ief_patch], loc='lower center', ncol=2,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Transformer Walk-Forward Policy Weights (Test Period 2024-07 -> 2026-04)', fontsize=11)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(FIG_DIR / 'weight_timeseries_05.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved weight_timeseries_05.png')

In [ ]:
# ── Figure 4: Seed Sharpe Distribution — Box Plots ──────────────────────────
config_labels = stability_df['config'].tolist()
per_seed_data = stability_df['per_seed_sharpes'].tolist()
feat_var_list = [FEATURE_PANELS.keys().__iter__().__next__()
                 if c.split('_', 2)[1] == 'daily' else 'dailyrp'
                 for c in config_labels]

# Re-derive feat_variant from config keys directly
feat_var_list = [c.split('_', 2)[1] for c in config_labels]

fig, ax = plt.subplots(figsize=(16, 5))
bp = ax.boxplot(per_seed_data, patch_artist=True, notch=False,
                medianprops={'color': 'black', 'lw': 1.5})
bm_sh = ann_sharpe(BENCHMARKS['RiskParity-21d'])
ax.axhline(bm_sh, color='red', ls='--', lw=1.2, label=f'RiskParity Sharpe={bm_sh:.2f}')
for patch, fv in zip(bp['boxes'], feat_var_list):
    patch.set_facecolor('#cccccc' if fv == 'daily' else '#ff9933')
    patch.set_alpha(0.7)
short_labels = [
    c.replace('daily', 'd').replace('dailyrp', 'drp').replace('crra_shrinkage', 'shrink')
    for c in config_labels
]
ax.set_xticks(range(1, len(config_labels) + 1))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('OOS Sharpe (per seed)')
ax.set_title('Per-Seed OOS Sharpe Distribution — Walk-Forward 18 Configs')
grey_patch   = mpatches.Patch(color='#cccccc', alpha=0.7, label='daily-only')
orange_patch = mpatches.Patch(color='#ff9933', alpha=0.7, label='daily+RP')
ax.legend(handles=[grey_patch, orange_patch, ax.get_lines()[0]], fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'seed_sharpe_dist_05.png', dpi=150)
plt.show()
print('Saved seed_sharpe_dist_05.png')

In [ ]:
# ── Figure 5: Feature Ablation — daily-only vs daily+RP ─────────────────────
arch_loss_combos = [(a, l) for a in ['MLP', 'LSTM', 'Transformer'] for l in LOSS_KEYS]
x = np.arange(len(arch_loss_combos))
width = 0.35
sharpes_d, sharpes_r, errs_d, errs_r = [], [], [], []
for arch, lv in arch_loss_combos:
    row_d = stability_df[(stability_df['arch'] == arch) & (stability_df['loss'] == lv) & (stability_df['feat_variant'] == 'daily-only')]
    row_r = stability_df[(stability_df['arch'] == arch) & (stability_df['loss'] == lv) & (stability_df['feat_variant'] == 'daily+RP')]
    sharpes_d.append(row_d['sharpe_mean'].values[0] if len(row_d) else np.nan)
    sharpes_r.append(row_r['sharpe_mean'].values[0] if len(row_r) else np.nan)
    errs_d.append(row_d['sharpe_std'].values[0] if len(row_d) else 0)
    errs_r.append(row_r['sharpe_std'].values[0] if len(row_r) else 0)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width/2, sharpes_d, width, yerr=errs_d, label='daily-only',
       color='#cccccc', edgecolor='black', capsize=4, alpha=0.9)
ax.bar(x + width/2, sharpes_r, width, yerr=errs_r, label='daily+RP',
       color='#ff9933', edgecolor='black', capsize=4, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([f'{a}\n{l[:6]}' for a, l in arch_loss_combos], fontsize=8)
ax.set_ylabel('OOS Sharpe (mean +/- 1sd across seeds)')
ax.set_title('Feature Ablation: Effect of Realized Power on Walk-Forward OOS Sharpe')
ax.legend(fontsize=9)
ax.axhline(0, color='black', lw=0.8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_ablation_05.png', dpi=150)
plt.show()
print('Saved feature_ablation_05.png')

In [ ]:
# ── Figure 6: Top-12 Strategies Bar Chart ────────────────────────────────────
top12 = comp_df.head(12)
family_colors = {'DL-daily+RP': '#ff9933', 'DL-daily-only': '#cccccc', 'Benchmark': '#333333'}

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(
    range(len(top12)), top12['sharpe'],
    color=[family_colors.get(f, '#aaaaaa') for f in top12['family']],
    edgecolor='black', alpha=0.85
)
for bar, (_, row) in zip(bars, top12.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{row["sharpe"]:.2f}\nDD={row["max_dd"]:.0%}', ha='center', va='bottom', fontsize=6.5)

short_labels = []
for s in top12['strategy']:
    parts = s.split('_', 2)
    if len(parts) == 3:
        a, fv, lv = parts
        short_labels.append(f'{a}\n{FEAT_DISPLAY.get(fv, fv)[:6]}|{lv[:4]}')
    else:
        short_labels.append(s)

ax.set_xticks(range(len(top12)))
ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('OOS Annualized Sharpe')
ax.set_title('Top-12 Walk-Forward DL Strategies — OOS Sharpe with Max Drawdown')
patches = [mpatches.Patch(color=c, alpha=0.85, label=l) for l, c in family_colors.items()]
ax.legend(handles=patches, fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'top_strategies_05.png', dpi=150)
plt.show()
print('Saved top_strategies_05.png')

## §11 Save Artifacts

In [ ]:
stab_save = stability_df.drop(columns=['per_seed_sharpes']).copy()
stab_save.to_csv(OUT_DIR / 'stability.csv', index=False)
print(f'Stability table:  {OUT_DIR / "stability.csv"}  ({len(stab_save)} rows)')

ret_df = pd.DataFrame(STRATEGY_RETURNS)
for bname, bret in BENCHMARKS.items():
    ret_df[bname] = bret
ret_df.to_parquet(OUT_DIR / 'strategy_returns.parquet')
print(f'Returns parquet:  shape={ret_df.shape}')

for name, wdf in STRATEGY_WEIGHTS.items():
    safe = name.replace('+', '')
    wdf.to_parquet(OUT_DIR / f'weights_{safe}.parquet')
print(f'Weight parquets:  {len(STRATEGY_WEIGHTS)} configs saved')
print('All artifacts saved.')

## §12 Headline + Findings

> **Off-axis comparability caveat.** The numbers in §12 are **not** comparable to the 29-asset leaderboard in Notebooks 00–04. This notebook uses a 2-asset universe (SPY + IEF) over a 2024-07 → 2026-04 test window with Risk Parity as the in-universe benchmark. Read **1.240 against 1.247** here — not against 2.579, 2.422, or 2.386. This notebook is a paradigm probe; it has no leaderboard rank.

In [ ]:
top_dl   = comp_df[comp_df['arch'] != 'Benchmark'].iloc[0]
rp_bmark = comp_df[comp_df['strategy'] == 'RiskParity-21d'].iloc[0]

abl = stability_df.copy()
daily_mean = abl[abl['feat_variant'] == 'daily-only']['sharpe_mean'].mean()
rp_mean    = abl[abl['feat_variant'] == 'daily+RP']['sharpe_mean'].mean()
delta_rp   = rp_mean - daily_mean

best_daily = abl[abl['feat_variant'] == 'daily-only'].sort_values('sharpe_mean', ascending=False).iloc[0]
best_rp    = abl[abl['feat_variant'] == 'daily+RP'].sort_values('sharpe_mean', ascending=False).iloc[0]

print('=' * 68)
print('  NOTEBOOK 05 — DIRECT-WEIGHT WALK-FORWARD DL PORTFOLIO (SPY+IEF)')
print('=' * 68)
print(f'  Run mode: {RUN_MODE}  |  Seeds: {len(SEEDS)}  |  Refits: {len(active_refit_dates)}')
print(f'  Top DL strategy:         {top_dl["strategy"]}')
print(f'    OOS Sharpe:            {top_dl["sharpe"]:.3f}')
print(f'    Ann. Return:           {top_dl["ann_ret"]:.1%}')
print(f'    Ann. Vol:              {top_dl["ann_vol"]:.1%}')
print(f'    Max Drawdown:          {top_dl["max_dd"]:.1%}')
print()
print(f'  Risk Parity benchmark:   Sharpe={rp_bmark["sharpe"]:.3f}')
print()
print(f'  Feature ablation (mean across arch x loss configs):')
print(f'    Daily-only Sharpe avg: {daily_mean:.3f}')
print(f'    Daily+RP Sharpe avg:   {rp_mean:.3f}')
print(f'    DeltaSharpe (RP):      {delta_rp:+.3f}')
print()
print(f'  Best daily-only:  {best_daily["config"]}  {best_daily["sharpe_mean"]:.3f}+/-{best_daily["sharpe_std"]:.3f}')
print(f'  Best daily+RP:    {best_rp["config"]}  {best_rp["sharpe_mean"]:.3f}+/-{best_rp["sharpe_std"]:.3f}')
print()
print(f'  Comparison rows: {len(comp_df)} (18 DL + 3 benchmarks)')
print(f'  Progress log:    {OUT_DIR / "config_progress.csv"}')
print('=' * 68)
print('  Populate docs/handoff/notebook_05_findings.md with the above.')
print('=' * 68)

## §13 Download Results (Colab Only)

In [ ]:
if IS_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive('results_nb05_jpmspec', 'zip', str(OUT_DIR))
    print('Triggering download of results_nb05_jpmspec.zip ...')
    files.download('results_nb05_jpmspec.zip')
    print('Done. Extract to ~/Projects/next-gen-aiam/results/notebook_05/ on your Mac.')
else:
    print(f'Local run complete. Results in: {OUT_DIR}')